In [1]:
import sys, os
sys.path.append(os.path.abspath("../../"))
from src.dataset.data_loader import load_competition_data

train_dataset, test_dataset, sample_submission = load_competition_data()
print("Train shape:", train_dataset.shape)
print("Test shape :", test_dataset.shape)

Train shape: (2000, 7)
Test shape : (500, 7)


In [2]:
from torch.utils.data import Dataset
import torch, warnings, logging
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer, default_data_collator
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPTIONS = ['A', 'B', 'C', 'D', 'E']
print(f"Device : {DEVICE}")

c:\Users\ais\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu


In [3]:
# HuggingFace Hub download bars + auth warning
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

# datasets library (.map(), load_dataset) progress bars
from datasets.utils.logging import disable_progress_bar as ds_disable_progress_bar
ds_disable_progress_bar()

# transformers logging (loading model/config messages)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

# general warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)

In [4]:
train_df = train_dataset.copy()
OPTIONS = ['A', 'B', 'C', 'D', 'E']
MC_MODEL_NAME = "bert-base-uncased"    # multiple choice model

train_df = train_dataset.copy()
test_df = test_dataset.copy()

# ------ Label encoding A=0, B=1, C=2 and so on -------
label_to_id = { opt: i for i, opt in enumerate(OPTIONS) }
id_to_label = { i: opt for opt, i in label_to_id.items() }

train_df['label'] = train_df['answer'].map(label_to_id)
print("Label mapping:", label_to_id)
print(train_df[['answer', 'label']].head())

train_df.columns

Label mapping: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
  answer  label
0      B      1
1      A      0
2      C      2
3      B      1
4      A      0


Index(['prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'label'], dtype='str')

In [5]:
# Tokenization for multiple choice models
# MC_model expects input shape -> batch_size * num_choices(options available) * sequence_length
# 5 options for a Q -> 5 sequences for each Q(row)

mc_tokenizer = AutoTokenizer.from_pretrained(MC_MODEL_NAME)
def format_option(prompt, option_text):
    return str(prompt) + " [SEP] " + str(option_text)

def format_row(row):
    return [format_option(row['prompt'], row[opt]) for opt in OPTIONS]

# Tokenization for multiple choice models
# MC_model expects input shape -> batch_size * num_choices(options available) * sequence_length
# 5 options for a Q -> 5 sequences for each Q(row)

mc_tokenizer = AutoTokenizer.from_pretrained(MC_MODEL_NAME)

def tokenize_mcq_row(row, tokenizer, max_length=128):
    """Tokenize all 5 options for a single row and reshape to [1, 5, max_length]."""
    formatted = format_row(row)
    encoded = tokenizer(
        formatted,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return {k: v.unsqueeze(0) for k,v in encoded.items()}

# example (testing function)
row0_encoded = tokenize_mcq_row(train_df.loc[0], mc_tokenizer, max_length=128)
print("input_ids shape:", row0_encoded['input_ids'].shape)
print("Second dimension (num choices):", row0_encoded['input_ids'].shape[1])

# [
#   [                                    # <- batch dim (1 question)
#     [101, 2054, 5320, ..., 0, 0, 0],  # option A -> 128 numbers
#     [101, 2054, 6721, ..., 0, 0, 0],  # option B -> 128 numbers
#     [101, 2054, 8934, ..., 0, 0, 0],  # option C -> 128 numbers
#     [101, 2054, 3312, ..., 0, 0, 0],  # option D -> 128 numbers
#     [101, 2054, 9987, ..., 0, 0, 0],  # option E -> 128 numbers
#   ]
# ]

# row0_encoded['input_ids'] is a single vector which has 5 vectors as its value and each vector represent encode of the options of length 128.
# for 2000 row training set -> (2000, 5, 128)

input_ids shape: torch.Size([1, 5, 128])
Second dimension (num choices): 5


In [6]:
def build_hf_mc_dataset(df, tokenizer, max_length=128):
    """Build a Hugging Face Dataset of tokenized multiple-choice examples from a dataframe."""
    records = []
    for _, row in df.iterrows():
        formatted = format_row(row)
        encoded = tokenizer(
            formatted,
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )
        records.append({
            "input_ids": encoded["input_ids"],           # [5, max_length]
            "attention_mask": encoded["attention_mask"],  # [5, max_length]
            "labels": int(row["label"]),
        })
    return Dataset.from_list(records)

hf_train_100 = build_hf_mc_dataset(train_df.head(100), mc_tokenizer, max_length=128)
print(hf_train_100)
print("input_ids shape (first item):", len(hf_train_100[0]['input_ids']), "x", len(hf_train_100[0]['input_ids'][0]))
print("Tokenized choices stored:", len(hf_train_100[0]['input_ids']))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})
input_ids shape (first item): 5 x 128
Tokenized choices stored: 5


In [7]:
def tokenize_mcq_batch(df, tokenizer, max_length=128):
    formatted = []
    n = len(df)

    for _, row in df.iterrows():
        formatted.extend(format_row(row))

    encoded = tokenizer(
        formatted,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    return {k: v.view(n, 5, max_length) for k,v in encoded.items()}


# batch size of 16 - encoded shape is (16, 5, 128)
batch16_encoded = tokenize_mcq_batch(
    train_df.head(16),
    mc_tokenizer,
    max_length=128
)

print("input_ids shape (first 16 rows):", batch16_encoded['input_ids'].shape)
total_token_positions = batch16_encoded['input_ids'].numel()
print("Total token positions:", total_token_positions)   # 16*5*128

input_ids shape (first 16 rows): torch.Size([16, 5, 128])
Total token positions: 10240


In [8]:
# AutoModelForMultipleChoice gives five logits correspond to 5 options

from transformers import AutoModelForMultipleChoice

mc_model = AutoModelForMultipleChoice.from_pretrained(MC_MODEL_NAME)
mc_model.to(DEVICE)
mc_model.eval()

row0_inputs = {k: v.to(DEVICE) for k, v in row0_encoded.items()}
with torch.no_grad():
    row0_outputs = mc_model(**row0_inputs)

row0_logits = row0_outputs.logits
print(" ")
print("Logits shape:", row0_logits.shape)
print("Logits produced per question:", row0_logits.shape[1])

row0_label = torch.tensor([train_df.loc[0, 'label']]).to(DEVICE)

with torch.no_grad():
    row0_loss_outputs = mc_model(**row0_inputs, labels=row0_label)

row0_loss = row0_loss_outputs.loss
print("Loss tensor:", row0_loss)
print("Loss tensor ndim:", row0_loss.dim())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3970.73it/s]


 
Logits shape: torch.Size([1, 5])
Logits produced per question: 5
Loss tensor: tensor(1.5873)
Loss tensor ndim: 0


In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

lora_mc_model = get_peft_model(mc_model, lora_config)
lora_mc_model.print_trainable_parameters()

trainable_params = sum(p.numel() for p in lora_mc_model.parameters() if p.requires_grad)
print("Trainable parameters:", trainable_params)

trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693
Trainable parameters: 295681


In [10]:
def build_hf_mc_dataset(df, tokenizer, max_length=128):
    """Build a Hugging Face Dataset of tokenized multiple-choice examples from a dataframe."""
    records = []
    for _, row in df.iterrows():
        formatted = format_row(row)
        encoded = tokenizer(
            formatted,
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )
        records.append({
            "input_ids": encoded["input_ids"],           # [5, max_length]
            "attention_mask": encoded["attention_mask"],  # [5, max_length]
            "labels": int(row["label"]),
        })
    return Dataset.from_list(records)

hf_train_100 = build_hf_mc_dataset(train_df.head(100), mc_tokenizer, max_length=128)
print(hf_train_100)
print("input_ids shape (first item):", len(hf_train_100[0]['input_ids']), "x", len(hf_train_100[0]['input_ids'][0]))
print("Tokenized choices stored:", len(hf_train_100[0]['input_ids']))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})
input_ids shape (first item): 5 x 128
Tokenized choices stored: 5


In [11]:
# ============================================================
# Q1 — Encoded numeric label for the row at index 150
# ============================================================

q1_answer = int(train_df.loc[150, 'label'])
print(f"Q1 — encoded label at row 150 ({train_df.loc[150, 'answer']!r}): {q1_answer}")
print(" ")

# ============================================================
# Q2 — Character length of the row-0 Option B formatted input
#      format: str(prompt) + " [SEP] " + str(option_B)
# ============================================================
q2_string = str(train_df.loc[0, 'prompt']) + " [SEP] " + str(train_df.loc[0, 'B'])
q2_answer = len(q2_string)
print(f"Q2 — formatted string: {q2_string!r}")
print(f"Q2 — character length: {q2_answer}")
print(" ")

# ======================================================================
# Q3 — Second dimension of the row-0 input_ids tensor, shape [1, 5, 128]
# ======================================================================
q3_row0_encoded = tokenize_mcq_row(train_df.loc[0], mc_tokenizer, max_length=128)
print("Q3 — input_ids shape:", q3_row0_encoded['input_ids'].shape)
q3_answer = q3_row0_encoded['input_ids'].shape[1]
print(f"Q3 — second dimension (num choices): {q3_answer}")
print(" ")

# ====================================================================================
# Q4 — Total token positions in the first-16-rows input_ids tensor, shape [16, 5, 128]
# ====================================================================================
q4_batch16_encoded = tokenize_mcq_batch(train_df.head(16), mc_tokenizer, max_length=128)
print("Q4 — input_ids shape:", q4_batch16_encoded['input_ids'].shape)
q4_answer = q4_batch16_encoded['input_ids'].numel()
print(f"Q4 — total token positions: {q4_answer}")
print(" ")

# ============================================================
# Q5 — Number of logits produced for one question, output shape [1, 5]
# ============================================================
q5_inputs = {k: v.to(DEVICE) for k, v in q3_row0_encoded.items()}
with torch.no_grad():
    q5_outputs = mc_model(**q5_inputs)
print("Q5 — logits shape:", q5_outputs.logits.shape)
q5_answer = q5_outputs.logits.shape[1]
print(f"Q5 — logits produced per question: {q5_answer}")
print(" ")

# ===============================================================
# Q6 — Number of dimensions of the scalar loss tensor for row 0
# ============================================================
q6_label = torch.tensor([train_df.loc[0, 'label']]).to(DEVICE)
with torch.no_grad():
    q6_outputs = mc_model(**q5_inputs, labels=q6_label)
print("Q6 — loss tensor:", q6_outputs.loss)
q6_answer = q6_outputs.loss.dim()
print(f"Q6 — loss tensor dimensions: {q6_answer}")
print(" ")

# ============================================================
# Q7 — Trainable parameters after applying LoRA
#      r=8, lora_alpha=16, target_modules=["query","value"], lora_dropout=0.1,
#      bias="none", task_type=TaskType.SEQ_CLS
# ============================================================
q7_base_model = AutoModelForMultipleChoice.from_pretrained(MC_MODEL_NAME)
q7_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

q7_lora_model = get_peft_model(q7_base_model, q7_lora_config)
q7_answer = sum(p.numel() for p in q7_lora_model.parameters() if p.requires_grad)
print(f"Q7 — trainable parameters: {q7_answer:,}")
print(" ")

# ========================================================================
# Q8 — Tokenized choices stored in input_ids for the first HF Dataset item
#      (first 100 rows of train.csv)
# =======================================================================
q8_hf_dataset = build_hf_mc_dataset(train_df.head(100), mc_tokenizer, max_length=128)
q8_input_ids_shape = (len(q8_hf_dataset[0]['input_ids']), len(q8_hf_dataset[0]['input_ids'][0]))
print("Q8 — input_ids shape (first item):", q8_input_ids_shape)
q8_answer = q8_input_ids_shape[0]
print(f"Q8 — tokenized choices stored: {q8_answer}")
print(" ")

# ========================================================================
# Q9 — Tiny LoRA fine-tuning: first 32 rows, max_length=64, batch_size=4,
#      gradient_accumulation_steps=1, max_steps=4 -> final global_step
# ============================================================
q9_tokenizer = AutoTokenizer.from_pretrained(MC_MODEL_NAME)
q9_base_model = AutoModelForMultipleChoice.from_pretrained(MC_MODEL_NAME)

q9_lora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["query", "value"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS,
)
q9_model = get_peft_model(q9_base_model, q9_lora_config)

q9_train_ds = build_hf_mc_dataset(train_df.head(32), q9_tokenizer, max_length=64)

q9_training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
)

q9_trainer = Trainer(
    model=q9_model,
    args=q9_training_args,
    train_dataset=q9_train_ds,
    data_collator=default_data_collator,
)

q9_train_result = q9_trainer.train()
q9_answer = q9_trainer.state.global_step
print(f"Q9 — final global_step: {q9_answer}")
print(" ")

# =============================================================================
# Q10 — Probability assigned to Option E for row 0 after fine-tuning (Q9 model)
# =============================================================================
q10_row0_encoded = tokenize_mcq_row(train_df.loc[0], q9_tokenizer, max_length=64)
q10_inputs = {k: v.to(DEVICE) for k, v in q10_row0_encoded.items()}

q9_model.eval()
with torch.no_grad():
    q10_outputs = q9_model(**q10_inputs)

q10_probs = F.softmax(q10_outputs.logits, dim=-1).squeeze(0)
q10_answer = round(q10_probs[label_to_id['E']].item(), 4)
print("Q10 — probabilities (A-E):", [round(p, 4) for p in q10_probs.tolist()])
print(f"Q10 — probability assigned to Option E: {q10_answer}")

Q1 — encoded label at row 150 ('C'): 2
 
Q2 — formatted string: "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement."
Q2 — character length: 407
 
Q3 — input_ids shape: torch.Size([1, 5, 128])
Q3 — second dimension (num choices): 5
 
Q4 — input_ids shape: torch.Size([16, 5, 128])
Q4 — total token positions: 10240
 
Q5 — logits shape: torch.Size([1, 5])
Q5 — logits produced per question: 5
 
Q6 — loss tensor: tensor(1.5873)
Q6 — loss tensor dimensions: 0
 


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2343.67it/s]


Q7 — trainable parameters: 295,681
 
Q8 — input_ids shape (first item): (5, 128)
Q8 — tokenized choices stored: 5
 


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4109.75it/s]


{'loss': '1.644', 'grad_norm': '3.043', 'learning_rate': '5e-05', 'epoch': '0.125'}
{'loss': '1.565', 'grad_norm': '2.861', 'learning_rate': '3.75e-05', 'epoch': '0.25'}
{'loss': '1.621', 'grad_norm': '2.761', 'learning_rate': '2.5e-05', 'epoch': '0.375'}
{'loss': '1.658', 'grad_norm': '3.002', 'learning_rate': '1.25e-05', 'epoch': '0.5'}
{'train_runtime': '28.35', 'train_samples_per_second': '0.564', 'train_steps_per_second': '0.141', 'train_loss': '1.622', 'epoch': '0.5'}
Q9 — final global_step: 4
 
Q10 — probabilities (A-E): [0.1989, 0.1989, 0.2015, 0.199, 0.2017]
Q10 — probability assigned to Option E: 0.2017


In [12]:
print("MILESTONE 4 Q/A — ANSWERS SUMMARY")
print("=" * 55)
print(f"  Q1  — encoded label, row 150            : {q1_answer}")
print(f"  Q2  — char length, row 0 Option B input  : {q2_answer}")
print(f"  Q3  — input_ids 2nd dimension            : {q3_answer}")
print(f"  Q4  — total token positions (16 rows)    : {q4_answer}")
print(f"  Q5  — logits produced per question       : {q5_answer}")
print(f"  Q6  — loss tensor dimensions              : {q6_answer}")
print(f"  Q7  — LoRA trainable parameters           : {q7_answer:,}")
print(f"  Q8  — tokenized choices stored            : {q8_answer}")
print(f"  Q9  — final global_step                  : {q9_answer}")
print(f"  Q10 — probability assigned to Option E    : {q10_answer}")

MILESTONE 4 Q/A — ANSWERS SUMMARY
  Q1  — encoded label, row 150            : 2
  Q2  — char length, row 0 Option B input  : 407
  Q3  — input_ids 2nd dimension            : 5
  Q4  — total token positions (16 rows)    : 10240
  Q5  — logits produced per question       : 5
  Q6  — loss tensor dimensions              : 0
  Q7  — LoRA trainable parameters           : 295,681
  Q8  — tokenized choices stored            : 5
  Q9  — final global_step                  : 4
  Q10 — probability assigned to Option E    : 0.2017
